In [1]:
from platform import python_version
python_version()
import numpy as np
import torch
import numpy as np
from arsf_envi_reader import envi_header
import shutil
import os

import json
import math
import affine
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.gridspec as gridspec
from osgeo import gdal,ogr,osr


import numpy as np
# import matplotlib.pyplot as plt
from scipy.optimize import curve_fit



from tqdm import tqdm
# import multiprocess as mp
from numpy import trapz

C:\Users\laral\AppData\Roaming\Python\Python39\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
torch.cuda.is_available()

True

In [3]:
get_ipython().run_line_magic('config', 'Completer.use_jedi = False')

# using AVIRIS-NG to calculate the srf

In [2]:
in_header = envi_header.find_hdr_file(r"D:\wenqu\aviris\site2a_aviris_ng_srf_data\ang20190704t193319rfl\ang20190704t193319_rfl_v2v2_img.hdr")
header_data = envi_header.read_hdr_file(in_header)

In [3]:
# Get wavelengths and convert to NumPy array
low_res_wavelengths = header_data['wavelength'].split(',')
low_res_wavelengths = [float(w) for w in low_res_wavelengths]
low_res_wavelengths = np.array(low_res_wavelengths)


low_res_fwhm = header_data['fwhm'].split(',')
low_res_fwhm = [float(w) for w in low_res_fwhm]
low_res_fwhm = np.array(low_res_fwhm)

In [4]:
def gauss(x, f, w):
    sigma = f/(2 * np.sqrt(2 *np.log(2)))
    y = np.exp((-(x-w)**2) / (2 * sigma**2))
    return y

In [2]:
high_res_img = gdal.Open(r'E:\wenqu\2024_data\site6_2024\multi_or1')
high_res_reflectance = gdal.Open(r'E:\wenqu\2024_data\site6_2024\multi_or1').ReadAsArray() 

high_res_reflectance.shape

(273, 4312, 10882)

In [6]:
high_res_wavelength = [float(b.split(" ")[0]) for b in high_res_img.GetMetadata().values() if b != "nm"]
high_res_wavelength.sort()
high_res_wavelength = np.array(high_res_wavelength)

In [7]:
gaussian_weights = []
for n in range(len(low_res_fwhm)):
#     print(n)
    a = []
    for i in high_res_wavelength:
        x = gauss(i, low_res_fwhm[n], low_res_wavelengths[n])
        if x < 0.5:
            x = 0
            a.append(x)
        else:
            a.append(x)
#     print(a)
    gaussian_weights.append(a)

In [8]:
gaussian_weights = np.array(gaussian_weights)
print(gaussian_weights.shape)

(425, 273)


In [3]:
bands, H, W = high_res_reflectance.shape
N = H * W
high_res_img_flatten = high_res_reflectance.reshape(bands, -1).T   

In [10]:
high_res_img_flatten.shape

(47520000, 273)

In [11]:
srf = gaussian_weights                             # (nb, 273)
wl  = high_res_wavelength   
denoms = np.trapz(srf, wl, axis=1) + 1e-12   # (273,)

In [12]:
device = torch.device("cuda")

gaussian_weights_tensor = torch.from_numpy(
    np.array(gaussian_weights).astype(np.float32)
).to(device)                                 # (425, 273)

high_res_img_flatten_tensor = torch.from_numpy(
    high_res_img_flatten.astype(np.float32)
).to(device)                                 # (1758084, 273)

denoms_tensor = torch.from_numpy(
    denoms.astype(np.float32)
).to(device)                                 # (425,)

wl_tensor = torch.from_numpy(
    high_res_wavelength.astype(np.float32)
).to(device)                                 # (273,)

N_pixels = high_res_img_flatten_tensor.shape[0]   # 注意这里是 0 

batch_pixels = []
out_dir = "site6_pixel2"
if os.path.exists(out_dir):
    shutil.rmtree(out_dir)
os.makedirs(out_dir)

save_every = 1_000_000

with torch.no_grad():
    for i in tqdm(range(N_pixels)):
        # 这一行是第 i 个像素的 273 band
        pixel_spec = high_res_img_flatten_tensor[i, :]         # (273,)

        # 425 条 SRF × 这个像素的光谱 → (425, 273)
        nomin_inputs = gaussian_weights_tensor * pixel_spec    # 自动广播

        # 对波长积分：每行一条 SRF → (425,)
        nomins = torch.trapz(nomin_inputs, wl_tensor, dim=1)   # (425,)

        # 归一化
        pixel_single = nomins / denoms_tensor                  # (425,)

        batch_pixels.append(pixel_single.cpu().numpy())

        if (i + 1) % save_every == 0:
            arr = np.array(batch_pixels, dtype=np.float32)
            np.save(f"{out_dir}/{i:010d}.npy", arr)
            batch_pixels = []

# 最后一批
if len(batch_pixels):
    arr = np.array(batch_pixels, dtype=np.float32)
    np.save(f"{out_dir}/{N_pixels-1:010d}.npy", arr)
    batch_pixels = []


100%|██████████████████████████████████████████████████████████████████| 47520000/47520000 [5:56:54<00:00, 2219.04it/s]


In [13]:
print(os.getcwd())

C:\Users\laral\OneDrive\Documents\GitHub\hytools


# reshape the convoluted back to images

In [4]:
import os
import glob
import numpy as np

pixel_dir = r"E:\wenqu\2024_data\site6_2024\site6_pixel1"

files = sorted(glob.glob(os.path.join(pixel_dir, "*.npy")))
print(files)

pixel_all = np.vstack([np.load(f) for f in files])
 
print("pixel_all:", pixel_all.shape)
# (N_pixels, nb)


['E:\\wenqu\\2024_data\\site6_2024\\site6_pixel1\\0000999999.npy', 'E:\\wenqu\\2024_data\\site6_2024\\site6_pixel1\\0001999999.npy', 'E:\\wenqu\\2024_data\\site6_2024\\site6_pixel1\\0002999999.npy', 'E:\\wenqu\\2024_data\\site6_2024\\site6_pixel1\\0003999999.npy', 'E:\\wenqu\\2024_data\\site6_2024\\site6_pixel1\\0004999999.npy', 'E:\\wenqu\\2024_data\\site6_2024\\site6_pixel1\\0005999999.npy', 'E:\\wenqu\\2024_data\\site6_2024\\site6_pixel1\\0006999999.npy', 'E:\\wenqu\\2024_data\\site6_2024\\site6_pixel1\\0007999999.npy', 'E:\\wenqu\\2024_data\\site6_2024\\site6_pixel1\\0008999999.npy', 'E:\\wenqu\\2024_data\\site6_2024\\site6_pixel1\\0009999999.npy', 'E:\\wenqu\\2024_data\\site6_2024\\site6_pixel1\\0010999999.npy', 'E:\\wenqu\\2024_data\\site6_2024\\site6_pixel1\\0011999999.npy', 'E:\\wenqu\\2024_data\\site6_2024\\site6_pixel1\\0012999999.npy', 'E:\\wenqu\\2024_data\\site6_2024\\site6_pixel1\\0013999999.npy', 'E:\\wenqu\\2024_data\\site6_2024\\site6_pixel1\\0014999999.npy', 'E:\\wenq

In [5]:
nb = pixel_all.shape[1]   

In [6]:
img_HWC = pixel_all.reshape(H, W, nb)
print("img_HWC shape:", img_HWC.shape)   # (H, W, nb)

img_HWC shape: (4312, 10882, 425)


# filter out non values

In [7]:
band_max = np.max(np.abs(img_HWC), axis=(0, 1)) 
eps = 1e-6
valid_band_mask = band_max > eps

print("Total bands:", img_HWC.shape[2])
print("Valid bands:", np.sum(valid_band_mask))

Total bands: 425
Valid bands: 122


In [8]:
img_HWC_filtered = img_HWC[:, :, valid_band_mask]
img_HWC_filtered.shape

(4312, 10882, 122)

In [9]:
H, W, nb_valid = img_HWC_filtered.shape
src_path = r'E:\wenqu\2024_data\site6_2024\multi_or1'
src_ds = gdal.Open(src_path)

geotransform = src_ds.GetGeoTransform()
projection   = src_ds.GetProjection()
src_ds = None  # close

# -------------------------------------------------
# 3. Create a new GeoTIFF and write bands
# -------------------------------------------------
out_path = r"E:\wenqu\2024_data\site6_2024\site6_ortho1.tif"

driver = gdal.GetDriverByName("GTiff")
# Create: cols, rows, bands, data type
out_ds = driver.Create(out_path, W, H, nb_valid, gdal.GDT_Float32)

# Set spatial reference
out_ds.SetGeoTransform(geotransform)
out_ds.SetProjection(projection)

# Write each band (GDAL uses band index starting at 1)
for b in range(nb_valid):
    band_array = img_HWC_filtered[:, :, b]    # (H, W)
    out_ds.GetRasterBand(b + 1).WriteArray(band_array)
    # Optional: set NoData
    # out_ds.GetRasterBand(b + 1).SetNoDataValue(-9999)

out_ds.FlushCache()
out_ds = None  # close and save

print("Saved filtered convolved image to:")
print(out_path)

Saved filtered convolved image to:
E:\wenqu\2024_data\site6_2024\site6_ortho1.tif


In [1]:
import os
import glob
import numpy as np
from osgeo import gdal,ogr,osr
pixel_dir = r"D:\wenqu\2024_data\convolution\site3_1"   # folder with many *.npy pixel files
src_path  = r"D:\wenqu\2024_data\site3\mosaic\multi_or1"  # original mosaic to copy georef from
out_path  = r"D:\wenqu\2024_data\convolution\site3_convoluted_1.tif"


In [2]:
files = sorted(glob.glob(os.path.join(pixel_dir, "*.npy")))
print("Found", len(files), "npy files")


Found 76 npy files


In [3]:
example = np.load(files[0])       # shape: (N_pixels_in_file, nb)
nb = example.shape[1]
print("Bands in original convolution:", nb)

Bands in original convolution: 425


In [ ]:
# ---------- 2. pass 1: find valid bands + total pixel count ----------
eps = 1e-6
band_max = np.zeros(nb, dtype=np.float32)
total_pixels = 0

for f in files:
    arr = np.load(f)             # (Ni, nb)
    total_pixels += arr.shape[0]
    band_max = np.maximum(band_max, np.max(np.abs(arr), axis=0))

valid_band_mask = band_max 
nb_valid = int(valid_band_mask.sum())

print("Total pixels:", total_pixels)
print("Valid bands:", nb_valid)

In [5]:
mm_path = os.path.join(pixel_dir, "pixel_all_valid")
pixel_all_valid = np.memmap(mm_path, dtype=np.float32,
                            mode='w+', shape=(total_pixels, nb_valid))

start = 0
for f in files:
    arr = np.load(f)                          # (Ni, nb)
    arr_valid = arr[:, valid_band_mask]       # keep only real bands
    end = start + arr_valid.shape[0]
    pixel_all_valid[start:end, :] = arr_valid
    start = end

pixel_all_valid.flush()

In [10]:
ds = gdal.Open(src_path)
W = ds.RasterXSize   # number of columns
H = ds.RasterYSize   # number of rows

print("Height (H):", H)
print("Width  (W):", W)

Height (H): 5730
Width  (W): 11212


In [11]:
assert H * W == total_pixels, "H*W must equal total number of pixels"

AssertionError: H*W must equal total number of pixels

In [8]:
img_HWC = pixel_all_valid.reshape(H, W, nb_valid)  # still memmap-backed

print("img_HWC shape:", img_HWC.shape)  # (H, W, nb_valid)

img_HWC shape: (5474, 10260, 19)


In [7]:
# ---------- 5. create GeoTIFF and write band by band ----------
src_ds = gdal.Open(src_path)
geotransform = src_ds.GetGeoTransform()
projection   = src_ds.GetProjection()
cols = src_ds.RasterXSize
rows = src_ds.RasterYSize
src_ds = None

assert rows == H and cols == W, "GeoTIFF size must match H, W"

driver = gdal.GetDriverByName("GTiff")
out_ds = driver.Create(out_path, cols, rows, nb_valid, gdal.GDT_Float32)

out_ds.SetGeoTransform(geotransform)
out_ds.SetProjection(projection)

for b in range(nb_valid):
    print(f"Writing band {b+1}/{nb_valid}")
    band_arr = np.array(img_HWC[:, :, b], dtype=np.float32)  # one band in memory
    out_ds.GetRasterBand(b + 1).WriteArray(band_arr)

out_ds.FlushCache()
out_ds = None

print("Done. Saved convoluted image to:", out_path)

NameError: name 'H' is not defined

# filter out non bands

In [ ]:
band_max = np.max(np.abs(img_HWC), axis=(0, 1)) 
eps = 1e-6
valid_band_mask = band_max > eps

print("Total bands:", img_HWC.shape[2])
print("Valid bands:", np.sum(valid_band_mask))


In [ ]:
img_HWC_filtered = img_HWC[:, :, valid_band_mask]
img_HWC_filtered.shape

In [ ]:
H, W, nb_valid = img_HWC_filtered.shape
src_path =r'E:\wenqu\2024_data\site3\mosaic\multi_or1'
src_ds = gdal.Open(src_path)

geotransform = src_ds.GetGeoTransform()
projection   = src_ds.GetProjection()
src_ds = None  # close

# -------------------------------------------------
# 3. Create a new GeoTIFF and write bands
# -------------------------------------------------
out_path = r"E:\wenqu\2024_data\site3\mosaic\simulation\site3_ortho1.tif"

driver = gdal.GetDriverByName("GTiff")
# Create: cols, rows, bands, data type
out_ds = driver.Create(out_path, W, H, nb_valid, gdal.GDT_Float32)

# Set spatial reference
out_ds.SetGeoTransform(geotransform)
out_ds.SetProjection(projection)

# Write each band (GDAL uses band index starting at 1)
for b in range(nb_valid):
    band_array = img_HWC_filtered[:, :, b]    # (H, W)
    out_ds.GetRasterBand(b + 1).WriteArray(band_array)
    # Optional: set NoData
    # out_ds.GetRasterBand(b + 1).SetNoDataValue(-9999)

out_ds.FlushCache()
out_ds = None  # close and save

print("Saved filtered convolved image to:")
print(out_path)

In [1]:
import os
import glob
import numpy as np
import spectral.io.envi as envi

# --------------------------------------------------
# 1) PATHS (EDIT THESE)
# --------------------------------------------------
# Your original ENVI dataset (either a folder containing .hdr, or the base path without .hdr)
src_path   = r"D:\wenqu\2024_data\site3\mosaic\multi_or3"

# Folder where your convolution chunks were saved (e.g., 0000000000.npy, 0001000000.npy, ...)
chunk_dir  = r"D:\wenqu\2024_data\convolution\site3_1"

# Output ENVI (base name, without extension)
out_dir    = r"D:\wenqu\2024_data\convolution"
out_base   = "site3_1"   # will write out_dir/site3_aviris425.hdr + .img
NBANDS_OUT = 425

os.makedirs(out_dir, exist_ok=True)

# --------------------------------------------------
# 2) FIND ORIGINAL .HDR AND READ SHAPE
# --------------------------------------------------
if os.path.isdir(src_path):
    hdr_list = glob.glob(os.path.join(src_path, "*.hdr"))
    if len(hdr_list) == 0:
        raise RuntimeError(f"No .hdr found in folder: {src_path}")
    hdr_path = hdr_list[0]
else:
    hdr_path = src_path if src_path.lower().endswith(".hdr") else (src_path + ".hdr")
    if not os.path.exists(hdr_path):
        raise RuntimeError(f"Could not find ENVI header: {hdr_path}")

orig = envi.open(hdr_path)
rows, cols, bands_in = orig.shape
print("Original ENVI shape (rows, cols, bands) =", (rows, cols, bands_in))

# --------------------------------------------------
# 3) LOAD + CONCATENATE ALL CHUNKS -> (N_pixels, 425)
# --------------------------------------------------
files = sorted([f for f in os.listdir(chunk_dir) if f.lower().endswith(".npy")])
if len(files) == 0:
    raise RuntimeError(f"No .npy chunks found in: {chunk_dir}")

all_pixels = []
for f in files:
    p = os.path.join(chunk_dir, f)
    arr = np.load(p)
    if arr.ndim != 2:
        raise ValueError(f"Chunk {f} must be 2D, got {arr.shape}")
    if arr.shape[1] != NBANDS_OUT:
        raise ValueError(f"Chunk {f} has {arr.shape[1]} bands, expected {NBANDS_OUT}")
    all_pixels.append(arr.astype(np.float32, copy=False))
    print("Loaded", f, arr.shape)

all_pixels = np.vstack(all_pixels).astype(np.float32)
print("Flat rebuilt array shape =", all_pixels.shape)

# --------------------------------------------------
# 4) SANITY CHECK PIXEL COUNT
# --------------------------------------------------
expected_pixels = rows * cols
if all_pixels.shape[0] != expected_pixels:
    raise RuntimeError(
        f"Pixel mismatch: expected rows*cols={expected_pixels}, got {all_pixels.shape[0]}"
    )

# --------------------------------------------------
# 5) RESHAPE BACK TO CUBE (rows, cols, 425)
# --------------------------------------------------
img_425 = all_pixels.reshape(rows, cols, NBANDS_OUT)
print("Final cube shape =", img_425.shape)

# --------------------------------------------------
# 6) WRITE ENVI OUTPUT
# --------------------------------------------------
out_hdr = os.path.join(out_dir, out_base + ".hdr")

envi.save_image(
    out_hdr,
    img_425,
    dtype=np.float32,
    interleave="bil",  # choose bil/bip/bsq; bil is common for hyperspectral
    force=True
)

print("Saved ENVI:", out_hdr)

Original ENVI shape (rows, cols, bands) = (5730, 11212, 273)
Loaded 0000999999.npy (1000000, 425)
Loaded 0001999999.npy (1000000, 425)
Loaded 0002999999.npy (1000000, 425)
Loaded 0003999999.npy (1000000, 425)
Loaded 0004999999.npy (1000000, 425)
Loaded 0005999999.npy (1000000, 425)
Loaded 0006999999.npy (1000000, 425)
Loaded 0007999999.npy (1000000, 425)
Loaded 0008999999.npy (1000000, 425)
Loaded 0009999999.npy (1000000, 425)
Loaded 0010999999.npy (1000000, 425)
Loaded 0011999999.npy (1000000, 425)
Loaded 0012999999.npy (1000000, 425)
Loaded 0013999999.npy (1000000, 425)
Loaded 0014999999.npy (1000000, 425)
Loaded 0015999999.npy (1000000, 425)
Loaded 0016999999.npy (1000000, 425)
Loaded 0017999999.npy (1000000, 425)
Loaded 0018999999.npy (1000000, 425)
Loaded 0019999999.npy (1000000, 425)
Loaded 0020999999.npy (1000000, 425)
Loaded 0021999999.npy (1000000, 425)
Loaded 0022999999.npy (1000000, 425)
Loaded 0023999999.npy (1000000, 425)
Loaded 0024999999.npy (1000000, 425)
Loaded 0025999

MemoryError: Unable to allocate 119. GiB for an array with shape (75146064, 425) and data type float32

In [ ]:
import os
import glob
import numpy as np
import spectral.io.envi as envi

# --------------------------------------------------
# 1) PATHS
# --------------------------------------------------
src_path   = r"D:\wenqu\2024_data\site3\mosaic\multi_or3"
chunk_dir  = r"D:\wenqu\2024_data\convolution\site3_1"
out_dir    = r"D:\wenqu\2024_data\convolution"
out_base   = "site3_1_filtered"
NBANDS_OUT = 425

os.makedirs(out_dir, exist_ok=True)

# --------------------------------------------------
# 2) READ ORIGINAL IMAGE SIZE
# --------------------------------------------------
if os.path.isdir(src_path):
    hdr_list = glob.glob(os.path.join(src_path, "*.hdr"))
    hdr_path = hdr_list[0]
else:
    hdr_path = src_path + ".hdr"

orig = envi.open(hdr_path)
rows, cols, bands_in = orig.shape
expected_pixels = rows * cols

print("Original ENVI shape:", rows, cols, bands_in)

# --------------------------------------------------
# 3) FIND VALID BANDS (streaming, 不占内存)
# --------------------------------------------------
files = sorted([f for f in os.listdir(chunk_dir) if f.endswith(".npy")])

valid_mask = np.zeros(NBANDS_OUT, dtype=bool)
seen_pixels = 0

for f in files:
    arr = np.load(os.path.join(chunk_dir, f), mmap_mode="r")

    n = min(arr.shape[0], expected_pixels - seen_pixels)
    if n <= 0:
        break

    arr = arr[:n]

    # 检查哪些 band 出现过非零值
    valid_mask |= np.any(arr != 0, axis=0)

    seen_pixels += n

    if valid_mask.all():
        break

valid_idx = np.where(valid_mask)[0]

print("Valid bands:", len(valid_idx))
print("Removed zero bands:", NBANDS_OUT - len(valid_idx))

# --------------------------------------------------
# 4) CREATE OUTPUT ENVI (only valid bands)
# --------------------------------------------------
out_hdr = os.path.join(out_dir, out_base + ".hdr")

meta = {
    "samples": cols,
    "lines": rows,
    "bands": len(valid_idx),
    "interleave": "bil",
    "data type": 4,
    "byte order": 0
}

envi.create_image(out_hdr, meta, ext=".dat", force=True)

out_img = envi.open(out_hdr)
mm = out_img.open_memmap(writable=True)

mm_flat = mm.reshape(-1, len(valid_idx))

# --------------------------------------------------
# 5) WRITE ONLY VALID BANDS (streaming)
# --------------------------------------------------
write_pos = 0

for f in files:
    arr = np.load(os.path.join(chunk_dir, f), mmap_mode="r")

    n = min(arr.shape[0], expected_pixels - write_pos)
    if n <= 0:
        break

    arr_valid = arr[:n, valid_idx]

    mm_flat[write_pos:write_pos+n] = arr_valid
    write_pos += n

    print(f"Writing {write_pos}/{expected_pixels}")

del mm_flat
del mm

print("✅ Finished writing filtered ENVI file")
print("Output:", out_hdr)

In [2]:
import os
import glob
import numpy as np
import spectral.io.envi as envi

# --------------------------------------------------
# 1) PATHS
# --------------------------------------------------
src_path   = r"D:\wenqu\2024_data\site3\mosaic\multi_or2"
chunk_dir  = r"D:\wenqu\2024_data\convolution\site3_pixel2"
out_dir    = r"D:\wenqu\2024_data\convolution"
out_base   = "site3_2"
NBANDS_OUT = 425

os.makedirs(out_dir, exist_ok=True)

# --------------------------------------------------
# 2) 读取原始 ENVI 尺寸
# --------------------------------------------------
if os.path.isdir(src_path):
    hdr_path = glob.glob(os.path.join(src_path, "*.hdr"))[0]
else:
    hdr_path = src_path + ".hdr"

orig = envi.open(hdr_path)
rows, cols, bands_in = orig.shape

print("Original shape:", rows, cols, bands_in)
expected_pixels = rows * cols

# --------------------------------------------------
# 3) 创建输出 ENVI (.dat)
# --------------------------------------------------
out_hdr = os.path.join(out_dir, out_base + ".hdr")

meta = {
    "samples": cols,
    "lines": rows,
    "bands": NBANDS_OUT,
    "interleave": "bil",
    "data type": 4,   # float32
    "byte order": 0
}

# 关键：ext='.dat'
envi.create_image(out_hdr, meta, ext=".dat", force=True)

print("Created:")
print(out_hdr)
print(out_hdr.replace(".hdr", ".dat"))

# --------------------------------------------------
# 4) 打开磁盘映射（memmap）
# --------------------------------------------------
out_img = envi.open(out_hdr)
mm = out_img.open_memmap(writable=True)

# 展平为 (N_pixels, 425)
mm_flat = mm.reshape(-1, NBANDS_OUT)

# --------------------------------------------------
# 5) 逐块写入（核心）
# --------------------------------------------------
files = sorted([f for f in os.listdir(chunk_dir) if f.endswith(".npy")])

write_pos = 0

for f in files:
    path = os.path.join(chunk_dir, f)
    arr = np.load(path, mmap_mode="r")  # 用 mmap 读取，避免吃内存

    n = arr.shape[0]

    if write_pos + n > expected_pixels:
        n = expected_pixels - write_pos
        arr = arr[:n]
        print("Truncated:", f)

    mm_flat[write_pos:write_pos+n] = arr
    write_pos += n

    print(f"Wrote {f} -> {write_pos}/{expected_pixels}")

    if write_pos >= expected_pixels:
        print("Pixel writing complete")
        break

# --------------------------------------------------
# 6) flush
# --------------------------------------------------
del mm_flat
del mm

print("✅ Finished writing ENVI .dat file")

Original shape: 5714 11912 273
Created:
D:\wenqu\2024_data\convolution\site3_2.hdr
D:\wenqu\2024_data\convolution\site3_2.dat
Wrote 0000999999.npy -> 1000000/68065168
Wrote 0001999999.npy -> 2000000/68065168
Wrote 0002999999.npy -> 3000000/68065168
Wrote 0003999999.npy -> 4000000/68065168
Wrote 0004999999.npy -> 5000000/68065168
Wrote 0005999999.npy -> 6000000/68065168
Wrote 0006999999.npy -> 7000000/68065168
Wrote 0007999999.npy -> 8000000/68065168
Wrote 0008999999.npy -> 9000000/68065168
Wrote 0009999999.npy -> 10000000/68065168
Wrote 0010999999.npy -> 11000000/68065168
Wrote 0011999999.npy -> 12000000/68065168
Wrote 0012999999.npy -> 13000000/68065168
Wrote 0013999999.npy -> 14000000/68065168
Wrote 0014999999.npy -> 15000000/68065168
Wrote 0015999999.npy -> 16000000/68065168
Wrote 0016999999.npy -> 17000000/68065168
Wrote 0017999999.npy -> 18000000/68065168
Wrote 0018999999.npy -> 19000000/68065168
Wrote 0019999999.npy -> 20000000/68065168
Wrote 0020999999.npy -> 21000000/68065168
W

In [1]:
import os
import glob
import numpy as np
import spectral.io.envi as envi

src_path  = r"D:\wenqu\2024_data\site3\mosaic\multi_or3"
chunk_dir = r"D:\wenqu\2024_data\convolution\site3_1"
out_dir   = r"D:\wenqu\2024_data\convolution"

out_name  = "site3_1"
NBANDS_OUT = 425

os.makedirs(out_dir, exist_ok=True)

In [2]:
if os.path.isdir(src_path):
    hdr = glob.glob(os.path.join(src_path, "*.hdr"))[0]
else:
    hdr = src_path + ".hdr"

img = envi.open(hdr)
rows, cols, bands = img.shape

print("原始影像尺寸:", rows, cols, bands)
print("总像素数:", rows * cols)

原始影像尺寸: 5730 11212 273
总像素数: 64244760


In [3]:
out_hdr = os.path.join(out_dir, out_name + ".hdr")

meta = {
    "samples": cols,
    "lines": rows,
    "bands": NBANDS_OUT,
    "interleave": "bil",
    "data type": 4,   # float32
    "byte order": 0
}

envi.create_image(out_hdr, meta, ext=".img", force=True)

	Data Source:   'D:\wenqu\2024_data\convolution\site3_1.img'
	# Rows:           5730
	# Samples:       11212
	# Bands:           425
	Interleave:        BIL
	Quantization:  32 bits
	Data format:   float32

In [4]:
out_img = envi.open(out_hdr)
mm = out_img.open_memmap(writable=True)

# 展平成 (N_pixels, 425)，方便按 pixel 顺序写
mm_flat = mm.reshape(-1, NBANDS_OUT)

print("输出 memmap shape:", mm_flat.shape)

输出 memmap shape: (64244760, 425)


In [5]:
files = sorted([f for f in os.listdir(chunk_dir) if f.endswith(".npy")])

write_pos = 0
expected_pixels = rows * cols

for f in files:
    path = os.path.join(chunk_dir, f)

    arr = np.load(path, mmap_mode="r")  # 关键：用 mmap 读取！
    n = arr.shape[0]

    # 防止 chunk 过多（你现在就是这个问题）
    if write_pos + n > expected_pixels:
        n = expected_pixels - write_pos
        arr = arr[:n]
        print("截断:", f)

    mm_flat[write_pos:write_pos+n] = arr
    write_pos += n

    print(f"写入 {f} -> 当前 {write_pos}/{expected_pixels}")

    if write_pos >= expected_pixels:
        print("像素已写满，停止")
        break

写入 0000999999.npy -> 当前 1000000/64244760
写入 0001999999.npy -> 当前 2000000/64244760
写入 0002999999.npy -> 当前 3000000/64244760
写入 0003999999.npy -> 当前 4000000/64244760
写入 0004999999.npy -> 当前 5000000/64244760
写入 0005999999.npy -> 当前 6000000/64244760
写入 0006999999.npy -> 当前 7000000/64244760
写入 0007999999.npy -> 当前 8000000/64244760
写入 0008999999.npy -> 当前 9000000/64244760
写入 0009999999.npy -> 当前 10000000/64244760
写入 0010999999.npy -> 当前 11000000/64244760
写入 0011999999.npy -> 当前 12000000/64244760
写入 0012999999.npy -> 当前 13000000/64244760
写入 0013999999.npy -> 当前 14000000/64244760
写入 0014999999.npy -> 当前 15000000/64244760
写入 0015999999.npy -> 当前 16000000/64244760
写入 0016999999.npy -> 当前 17000000/64244760
写入 0017999999.npy -> 当前 18000000/64244760
写入 0018999999.npy -> 当前 19000000/64244760
写入 0019999999.npy -> 当前 20000000/64244760
写入 0020999999.npy -> 当前 21000000/64244760
写入 0021999999.npy -> 当前 22000000/64244760
写入 0022999999.npy -> 当前 23000000/64244760
写入 0023999999.npy -> 当前 24000000/64244760
写

In [6]:
del mm_flat
del mm

print("ENVI 输出完成:", out_hdr)

ENVI 输出完成: D:\wenqu\2024_data\convolution\site3_1.hdr


In [1]:
import os
import glob
import numpy as np
import spectral.io.envi as envi

# --------------------------------------------------
# 1️⃣ 路径
# --------------------------------------------------
src_path  = r"D:\wenqu\2024_data\site3\mosaic\multi_or3"
chunk_dir = r"D:\wenqu\2024_data\convolution\site3_1"
out_dir   = r"D:\wenqu\2024_data\convolution"

out_name  = "site3_1"   # 输出文件名 (site3_1.hdr + site3_1.dat)
NBANDS_OUT = 425

os.makedirs(out_dir, exist_ok=True)

# --------------------------------------------------
# 2️⃣ 读取原始 ENVI 尺寸
# --------------------------------------------------
if os.path.isdir(src_path):
    hdr = glob.glob(os.path.join(src_path, "*.hdr"))[0]
else:
    hdr = src_path + ".hdr"

img = envi.open(hdr)
rows, cols, bands = img.shape

print("原始尺寸:", rows, cols, bands)
print("总像素:", rows * cols)

expected_pixels = rows * cols

# --------------------------------------------------
# 3️⃣ 创建输出 ENVI（注意 ext='.dat'）
# --------------------------------------------------
out_hdr = os.path.join(out_dir, out_name + ".hdr")

meta = {
    "samples": cols,
    "lines": rows,
    "bands": NBANDS_OUT,
    "interleave": "bil",   # 必须和你 flatten 顺序一致（你现在是 bil-safe）
    "data type": 4,        # float32
    "byte order": 0
}

envi.create_image(out_hdr, meta, ext=".dat", force=True)

print("已创建输出文件:")
print(out_hdr)
print(out_hdr.replace(".hdr", ".dat"))

# --------------------------------------------------
# 4️⃣ 打开 memmap（磁盘映射，不占内存）
# --------------------------------------------------
out_img = envi.open(out_hdr)
mm = out_img.open_memmap(writable=True)

# 展平成 (N_pixels, 425) 方便按像素写
mm_flat = mm.reshape(-1, NBANDS_OUT)

print("输出 memmap shape:", mm_flat.shape)

# --------------------------------------------------
# 5️⃣ 逐块写入（不会爆内存）
# --------------------------------------------------
files = sorted([f for f in os.listdir(chunk_dir) if f.endswith(".npy")])

write_pos = 0

for f in files:
    path = os.path.join(chunk_dir, f)

    arr = np.load(path, mmap_mode="r")   # 关键：mmap 读取
    n = arr.shape[0]

    # 防止 chunk 多于真实像素（你之前就发生了）
    if write_pos + n > expected_pixels:
        n = expected_pixels - write_pos
        arr = arr[:n]
        print("截断:", f)

    mm_flat[write_pos:write_pos+n] = arr
    write_pos += n

    print(f"写入 {f} -> {write_pos}/{expected_pixels}")

    if write_pos >= expected_pixels:
        print("像素写满，停止")
        break

# --------------------------------------------------
# 6️⃣ flush 到磁盘
# --------------------------------------------------
del mm_flat
del mm

print("✅ 完成写入 ENVI .dat 文件")

原始尺寸: 5730 11212 273
总像素: 64244760
已创建输出文件:
D:\wenqu\2024_data\convolution\site3_1.hdr
D:\wenqu\2024_data\convolution\site3_1.dat
输出 memmap shape: (64244760, 425)
写入 0000999999.npy -> 1000000/64244760
写入 0001999999.npy -> 2000000/64244760
写入 0002999999.npy -> 3000000/64244760
写入 0003999999.npy -> 4000000/64244760
写入 0004999999.npy -> 5000000/64244760
写入 0005999999.npy -> 6000000/64244760
写入 0006999999.npy -> 7000000/64244760
写入 0007999999.npy -> 8000000/64244760
写入 0008999999.npy -> 9000000/64244760
写入 0009999999.npy -> 10000000/64244760
写入 0010999999.npy -> 11000000/64244760
写入 0011999999.npy -> 12000000/64244760
写入 0012999999.npy -> 13000000/64244760
写入 0013999999.npy -> 14000000/64244760
写入 0014999999.npy -> 15000000/64244760
写入 0015999999.npy -> 16000000/64244760
写入 0016999999.npy -> 17000000/64244760
写入 0017999999.npy -> 18000000/64244760
写入 0018999999.npy -> 19000000/64244760
写入 0019999999.npy -> 20000000/64244760
写入 0020999999.npy -> 21000000/64244760
写入 0021999999.npy -> 220000